# PEFT full run

| Phase | Runtime | What runs
|------|---------|-----------
| 1 | **generation** (`requirements.txt`) | `peft_sweep` -- train all 4 cells x 3 epochs, eval_loss pre-filter, generate + score val candidates
| 2 | **COMET** (`requirements-comet.txt`) | `peft_verify` (COMET + judge phi), **freeze** the adapter
| 3 | **generation** (`requirements.txt`) | frozen `peft` condition on val + chrF/BLEU + stylometrics + judge phi 
| 4 | **COMET** (`requirements-comet.txt`) | `peft` COMET + paired bootstrap vs the ladder 


---
## Phase 1 — generation runtime · the sweep


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

In [ ]:
# Generation stack.
!pip install -r requirements.txt

!pip uninstall -y torchvision torchaudio

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

### Persist the expensive artifacts across sessions


In [ ]:
PERSIST = True
DRIVE_ROOT = '/content/drive/MyDrive/style-aware-mt/peft'

import os, pathlib, shutil
if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    for rel in ('models', 'outputs/peft_sweep', 'results_backup'):
        target = pathlib.Path(DRIVE_ROOT) / rel
        target.mkdir(parents=True, exist_ok=True)
        if rel == 'results_backup':
            continue                                  # backup dir only, not linked in
        link = pathlib.Path(rel)
        if link.is_symlink():
            print(f'{link} -> {link.resolve()} (already linked)')
            continue
        if link.exists():
            # outputs/peft_sweep is a real, committed dir. Move what is in it onto
            # Drive and replace it with the link, or nothing written there persists.
            moved = 0
            for item in link.iterdir():
                dest = target / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
                    moved += 1
            shutil.rmtree(link)
            print(f'{link}: moved {moved} existing file(s) onto Drive, replacing dir with link')
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(target, target_is_directory=True)
        print(f'{link} -> {target}')
    !ls -la models outputs/peft_sweep | head -20
    # 4 cells x 3 epoch checkpoints x ~400 MB (81 MB adapter + ~320 MB optimizer
    # state) is 5-6 GB. Check the headroom now: filling Drive mid-training kills
    # the cell in flight and the manifest never gets written.
    print()
    !df -h /content/drive/MyDrive | tail -1
else:
    print('PERSIST=False -- artifacts live on the VM only. Phase 1 alone is ~16 h against a '
          '12 h session cap, so this WILL lose training you have already paid for. Only '
          'sensible on a machine with no session limit.')


### Preflight: clear the tiny-smoke artifacts out of the way

In [ ]:
import json, pathlib

VAL_N = sum(1 for line in open('data/splits/val.jsonl', encoding='utf-8') if line.strip())
print(f'full val = {VAL_N} segments\n')

for p in sorted(pathlib.Path('outputs/peft_sweep').glob('*_val.jsonl')):
    n = sum(1 for line in p.open(encoding='utf-8') if line.strip())
    if n < VAL_N:
        p.unlink()
        print(f'removed {p} ({n} rows — smoke/partial, would be reused as a candidate)')
    else:
        print(f'kept    {p} ({n} rows — full val)')

res = pathlib.Path('results/peft_sweep_val.json')
if res.exists():
    smoke = [c for c in json.loads(res.read_text(encoding='utf-8'))['cells'] if c['n'] < VAL_N]
    if smoke:
        res.unlink()
        print(f"removed {res} (smoke ranking over {[c['tag'] for c in smoke]}, n<{VAL_N})")
    else:
        print(f'kept    {res} (full-val ranking)')

### What is already trained (resume check)

In [ ]:
import json, yaml
from src.peft.sweep import _cell_dir          # the driver's own path rule, so no drift

# Reference only: the recorded outcome of the earlier multi-epoch run on a 4090
# (peft_multiepoch_smoke_colab.ipynb). The anchor cell is retrained from scratch in
# this run, and on different hardware it will not reproduce these exactly. Printed
# for orientation, never asserted.
ANCHOR_REFERENCE = [(1, 679, 1.5333), (2, 1358, 1.5591), (3, 2037, 1.7799)]

cfg = yaml.safe_load(open('configs/peft_sweep.yaml', encoding='utf-8'))
grid = cfg['sweep']['grid']
output_base = cfg['sweep'].get('output_base', 'models')

trained, partial, todo = [], [], []
for cell in grid:
    d = _cell_dir(output_base, int(cell['r']), float(cell['lr']))
    if (d / 'epoch_checkpoints.json').exists():
        man = json.loads((d / 'epoch_checkpoints.json').read_text(encoding='utf-8'))
        got = [(m['epoch'], m['step'], round(m['eval_loss'], 4)) for m in man]
        trained.append((d.name, got, bool(cell.get('anchor'))))
    elif d.exists() and any(d.iterdir()):
        # Checkpoints but no manifest: training was cut off mid-cell. The driver
        # only skips a cell whose manifest exists, so this one is retrained whole.
        partial.append(d.name)
    else:
        todo.append(d.name)

for name, got, is_anchor in trained:
    print(f'trained   {name}  {got}')
    if is_anchor:
        print(f'          4090 reference: {ANCHOR_REFERENCE}')
for name in partial:
    print(f'PARTIAL   {name} -- checkpoints but no manifest; will be retrained from scratch')
for name in todo:
    print(f'to train  {name}')

remaining = len(partial) + len(todo)
print(f'\n{len(trained)}/{len(grid)} cells complete; {remaining} to train '
      f'(~3 h each on an A100 = ~{remaining * 3} h of training ahead).')


### The grid plan


In [ ]:
!python manage.py peft_sweep --config configs/peft_sweep.yaml --dry-run

### Train, pre-filter, generate, score

In [ ]:
!python manage.py peft_sweep --config configs/peft_sweep.yaml --epochs-keep 2

### Manifest check on all four trained cells


In [ ]:
import json
from pathlib import Path

# Every cell was trained in this run, so every cell gets the same property checks:
# three distinct epoch checkpoints, eval_loss actually moving, weights on disk.
manifests = sorted(Path('models').glob('peft_lora_*/epoch_checkpoints.json'))
print(f'{len(manifests)} cell manifest(s) found\n')
problems = []
for man_path in manifests:
    name = man_path.parent.name
    man = json.loads(man_path.read_text(encoding='utf-8'))
    epochs = [m['epoch'] for m in man]
    steps = [m['step'] for m in man]
    losses = [m['eval_loss'] for m in man]
    missing = [m['checkpoint'] for m in man if not m['checkpoint'] or not Path(m['checkpoint']).exists()]
    print(f'{name:<26} epochs={epochs} steps={steps}')
    print('    eval_loss: ' + '  '.join(f'e{e}={l:.4f}' for e, l in zip(epochs, losses)))
    if epochs != [1, 2, 3]:
        problems.append(f'{name}: epochs not 1..3 -> {epochs}')
    if len(set(steps)) != len(steps):
        problems.append(f'{name}: duplicate checkpoint step -> {steps}')
    if len(set(round(l, 6) for l in losses)) <= 1:
        problems.append(f'{name}: eval_loss flat across epochs -> {losses}')
    if missing:
        problems.append(f'{name}: checkpoint dir(s) missing on disk -> {missing}')

assert len(manifests) == 4, f'expected 4 cells present, found {len(manifests)}'
assert not problems, 'manifest problems:\n  ' + '\n  '.join(problems)
print('\nMANIFESTS OK: 4 cells, 3 distinct epoch checkpoints each, losses moving, all on disk.')

# Soft comparison of the retrained anchor against the 4090 run. Different GPU, so
# some drift is expected; this is orientation, not a gate.
anchor = Path('models/peft_lora_r16_lr2e-4/epoch_checkpoints.json')
if anchor.exists():
    got = [(m['epoch'], m['step'], round(m['eval_loss'], 4))
           for m in json.loads(anchor.read_text(encoding='utf-8'))]
    deltas = [round(a[2] - b[2], 4) for a, b in zip(got, ANCHOR_REFERENCE)]
    print(f'\nanchor cell here : {got}')
    print(f'4090 reference   : {ANCHOR_REFERENCE}')
    print(f'eval_loss delta  : {deltas}'
          + ('   <-- large; check seed / data / precision' if max(map(abs, deltas)) > 0.1 else ''))


### Optional: drop optimizer state once a cell is done


In [ ]:
from pathlib import Path

DROP = False        # flip to True to actually delete

removed, freed = [], 0
for f in sorted(Path('models').glob('peft_lora_*/checkpoint-*/optimizer.pt')):
    size = f.stat().st_size
    freed += size
    removed.append(f)
    if DROP:
        f.unlink()
print(f'{len(removed)} optimizer state file(s), {freed / 1e9:.2f} GB'
      + (' deleted' if DROP else ' found (DROP=False, nothing deleted)'))
for f in removed[:12]:
    print('  ', f)


### The proxy pick


In [ ]:
import json

sweep = json.load(open('results/peft_sweep_val.json'))
print(f"epochs_keep={sweep['epochs_keep']}  adequacy_margin={sweep['adequacy_margin']}  "
      f"candidates={len(sweep['cells'])}\n")
hdr = f"{'tag':<24}{'r':>4}{'a':>5}{'lr':>9}{'ep':>4}{'n':>6}{'chrF':>8}{'reg_fit':>9}{'eval_loss':>11}"
print(hdr); print('-' * len(hdr))
for c in sorted(sweep['cells'], key=lambda c: c.get('register_fit', 1e9)):
    print(f"{c['tag']:<24}{c['r']:>4}{c['alpha']:>5}{c['lr']:>9g}{c['epoch']:>4}{c['n']:>6}"
          f"{c['chrF']:>8}{c.get('register_fit', float('nan')):>9}{c['eval_loss']:>11.4f}")

rec = sweep.get('recommended')
print('\nproxy recommended:', rec and {k: rec[k] for k in ('tag', 'r', 'lr', 'epoch', 'chrF', 'register_fit') if k in rec})

# Every candidate must have been generated on FULL val, not a leftover short file.
short = [c['tag'] for c in sweep['cells'] if c['n'] != VAL_N]
assert not short, f'candidates not scored on full val ({VAL_N}): {short}'

In [ ]:
# Back the phase-1 result up to Drive before switching runtimes.
if PERSIST:
    !cp -v results/peft_sweep_val.json {DRIVE_ROOT}/results_backup/

---
## Phase 2 — COMET runtime · verify + freeze


In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:

!USE_TF=0 python -m src.peft.verify --config configs/peft_sweep.yaml \
    --judge-config configs/judge_eval.yaml --top 3

In [ ]:
# The freeze decision from the reported metrics.
import json

v = json.load(open('results/peft_verify_val.json'))
print('freeze tag :', v['freeze'],
      '(proxy pick held)' if v['proxy_pick_held'] else '(runner-up overtook the proxy pick)')
print('checkpoint :', v['freeze_checkpoint'], '\n')
for c in v['cells']:
    mark = '  <== freeze' if c['tag'] == v['freeze'] else ''
    phi = f"{c['judge_mean']:.3f}" if c['judge_mean'] is not None else 'n/a'
    print(f"  {c['tag']:<24} r={c['r']} lr={c['lr']:g} ep={c['epoch']}  "
          f"chrF {c['chrF']}  COMET {c['comet_system']:.4f}  Phi {phi}"
          f"  (judge cov {c['judge_coverage']}){mark}")

### Freeze the adapter into `configs/peft_qwen.yaml`

`generator.adapter_path` is what the `peft` inference condition loads. Point it
at the frozen checkpoint before generating the reported condition.

In [ ]:
import json, re, pathlib

v = json.load(open('results/peft_verify_val.json'))
ckpt = v['freeze_checkpoint']
assert ckpt and pathlib.Path(ckpt).exists(), f'frozen checkpoint missing on disk: {ckpt}'
frozen = next(c for c in v['cells'] if c['tag'] == v['freeze'])

p = pathlib.Path('configs/peft_qwen.yaml')
text = p.read_text(encoding='utf-8')
text, n = re.subn(r'(?m)^(\s*adapter_path:\s*)\S+', lambda m: f'{m.group(1)}{ckpt}', text, count=1)
assert n == 1, 'no generator.adapter_path line found in configs/peft_qwen.yaml'
p.write_text(text, encoding='utf-8')

print(f"froze generator.adapter_path = {ckpt}")
print(f"  (r={frozen['r']}, alpha={frozen['alpha']}, lr={frozen['lr']:g}, epoch={frozen['epoch']})")
!grep -n 'adapter_path:' configs/peft_qwen.yaml

In [ ]:
if PERSIST:
    !cp -v results/peft_verify_val.json {DRIVE_ROOT}/results_backup/

---
## Phase 3 — generation runtime · the frozen `peft` condition

In [ ]:
!pip install -q -r requirements.txt
# Same torchvision/torchaudio ABI mismatch as Phase 1 — drop them (text-only pipeline).
!pip uninstall -y torchvision torchaudio

In [ ]:
# Full val with the frozen adapter; resumable via outputs/peft_val.jsonl.
!python manage.py infer --condition peft --config configs/peft_qwen.yaml

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics — free/local, no COMET.
!python manage.py eval         --conditions peft --split val
!python manage.py stylometrics --conditions peft --split val

In [ ]:
# Register fidelity (judge Phi) for the reported PEFT row.
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
!python manage.py judge --conditions peft --split val --config configs/judge_eval.yaml

---
## Phase 4 — COMET runtime · PEFT COMET + paired bootstrap


In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
!python manage.py comet --conditions peft --split val

In [ ]:
import pathlib

LADDER = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full']
present = [c for c in LADDER if pathlib.Path(f'outputs/{c}_val.jsonl').exists()]
absent = [c for c in LADDER if c not in present]
print('ladder outputs present:', present)
if absent:
    print('MISSING (excluded from the bootstrap):', absent)

In [ ]:
# PEFT vs the ladder rungs that are present, baseline = knn_fewshot when available.
conds = ' '.join(['peft'] + present)
baseline = 'knn_fewshot' if 'knn_fewshot' in present else 'peft'
!python manage.py bootstrap --metric comet --conditions {conds} --split val --baseline {baseline}

In [ ]:
if PERSIST:
    !cp -v results/*val*.json {DRIVE_ROOT}/results_backup/ 2>/dev/null; ls {DRIVE_ROOT}/results_backup